# Ultimate NIDS Pipeline: Supervised Vector Space Engineering
**Goal:** Fix the "Zero Recall" on R2L/U2R by warping the feature space to maximize class separation.

**The "Unconventional" Approach: NCA + Isolation Embeddings**
We upgrade from linear projections to non-linear Metric Learning and Explicit Vector Isolation.

**New Architecture:**
1.  **Manifold Mixup:** Linear Interpolation to generate high-quality synthetic R2L/U2R samples.
2.  **Neighborhood Components Analysis (NCA):** A powerful Metric Learning algorithm that learns a vector space where same-class points are spatially close, handling complex clusters (like DoS) better than LDA.
3.  **Isolation Embeddings:** We train separate Isolation Forests for each class. The anomaly scores from these forests become new axes in our vector space, providing explicit "Membership Probability" coordinates.
4.  **Deep MLP + Risk-Sensitive Inference:** The final classifier uses this hyper-enriched space.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc  # Garbage Collector for memory management

# The New Stack: NCA, Isolation Forests & Metric Learning
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, pairwise_distances

# Config
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

## 1. Data Loading

In [2]:
DATA_DIR = 'Data'
CSV_FILE = os.path.join(DATA_DIR, 'network_connections.csv')
MAP_FILE = os.path.join(DATA_DIR, 'attack2category_map.txt')

# 1. Load Mapping
attack_map = {'normal': 'normal'}
try:
    with open(MAP_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                attack_map[parts[0]] = parts[1]
except FileNotFoundError:
    print("Warning: Map file not found. Creating dummy map.")

# 2. Load Data
df = pd.read_csv(CSV_FILE)
df['label'] = df['label'].astype(str).str.replace('.', '', regex=False)
df['category'] = df['label'].map(attack_map).fillna('other')
df.drop_duplicates(inplace=True)
print(f"Data Loaded. Shape: {df.shape}")

Data Loaded. Shape: (125973, 43)


## 2. Vector Space Preparation
We use `float32` to save memory.

In [3]:
def prepare_vector_input(data, fit=False, encoders=None):
    df_vec = data.copy()
    
    # 1. Log Transform
    for col in ['src_bytes', 'dst_bytes', 'duration']:
        if col in df_vec.columns:
            df_vec[col] = np.log1p(df_vec[col]).astype(np.float32)

    # 2. Categorical Handling: Frequency Encoding 
    cat_cols = ['protocol_type', 'service', 'flag']
    new_encoders = {}
    
    for col in cat_cols:
        if fit:
            freq_map = df_vec[col].value_counts(normalize=True).to_dict()
            df_vec[col] = df_vec[col].map(freq_map).astype(np.float32)
            new_encoders[col] = freq_map
        else:
            df_vec[col] = df_vec[col].map(encoders[col]).fillna(0).astype(np.float32)
            
    return df_vec, new_encoders

# Apply Preparation
X = df.drop(['label', 'category'], axis=1)
y = df['category']

X_vec, encoders = prepare_vector_input(X, fit=True)
y_vec = LabelEncoder().fit_transform(y)

# Clean up raw DF to save memory
del df, X, y
gc.collect()

# Split
X_train, X_test, y_train, y_test = train_test_split(X_vec, y_vec, test_size=0.2, stratify=y_vec, random_state=42)

print("Data vectorized and memory cleaned.")

Data vectorized and memory cleaned.


## 3. Manifold Mixup (Advanced Oversampling)
Linear Interpolation to creating synthetic data.

In [4]:
print("--- Performing Manifold Mixup Oversampling ---")

train_df = X_train.copy()
train_df['target'] = y_train

dfs = []
for cls in train_df['target'].unique():
    cls_df = train_df[train_df['target'] == cls]
    count = len(cls_df)
    
    # Aggressive Target for Rare Classes
    TARGET_COUNT = 4000
    
    if count < TARGET_COUNT:
        needed = TARGET_COUNT - count
        indices = np.random.choice(cls_df.index, needed, replace=True)
        indices2 = np.random.choice(cls_df.index, needed, replace=True)
        
        part1 = cls_df.loc[indices].drop('target', axis=1).reset_index(drop=True)
        part2 = cls_df.loc[indices2].drop('target', axis=1).reset_index(drop=True)
        
        alpha = np.random.uniform(0, 1, size=(needed, 1)).astype(np.float32)
        synthetic = part1 * alpha + part2 * (1 - alpha)
        synthetic['target'] = cls
        
        print(f" > Manifold Mixup Class {cls}: {count} original + {needed} synthetic")
        dfs.append(cls_df)
        dfs.append(synthetic)
    else:
        dfs.append(cls_df.sample(n=min(count, 10000), random_state=42))

train_balanced = pd.concat(dfs).sample(frac=1, random_state=42)
X_train_bal = train_balanced.drop('target', axis=1)
y_train_bal = train_balanced['target']

# Clean up
del train_df, dfs
gc.collect()

print(f"Balanced Training Shape: {X_train_bal.shape}")

--- Performing Manifold Mixup Oversampling ---
 > Manifold Mixup Class 3: 796 original + 3204 synthetic
 > Manifold Mixup Class 4: 42 original + 3958 synthetic
Balanced Training Shape: (37325, 41)


## 4. Vector Space Engineering (NCA + Isolation Embeddings)
**FIX:** We set `n_jobs=1` for Isolation Forest to prevent process spawning crashes.

In [5]:
print("--- Engineering Vector Space (NCA + Isolation) ---")

# 1. Standard Scaler (Base)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal).astype(np.float32)
X_test_sc = scaler.transform(X_test).astype(np.float32)

# 2. Neighborhood Components Analysis (NCA)
print("Fitting NCA (Metric Learning)...")
nca = NeighborhoodComponentsAnalysis(n_components=15, random_state=42)
# Downsample for fitting NCA transform
idx_sample = np.random.choice(len(X_train_sc), size=min(20000, len(X_train_sc)), replace=False)
nca.fit(X_train_sc[idx_sample], y_train_bal.iloc[idx_sample])

X_train_nca = nca.transform(X_train_sc).astype(np.float32)
X_test_nca = nca.transform(X_test_sc).astype(np.float32)

# 3. Isolation Embeddings (The "Vector Isolating" Method)
print("Generating Isolation Embeddings...")
iso_feats_train = []
iso_feats_test = []
unique_classes = np.sort(train_balanced['target'].unique())
iso_models = {} # Store models for reuse in validation

for cls in unique_classes:
    X_cls = X_train_sc[y_train_bal == cls]
    if len(X_cls) < 50: continue
    
    # CRITICAL FIX: n_jobs=1 prevents kernel crashes in loops
    iso = IsolationForest(n_estimators=100, contamination=0.01, n_jobs=1, random_state=42)
    iso.fit(X_cls)
    iso_models[cls] = iso # Save model
    
    score_train = iso.decision_function(X_train_sc)
    score_test = iso.decision_function(X_test_sc)
    
    iso_feats_train.append(score_train.reshape(-1, 1).astype(np.float32))
    iso_feats_test.append(score_test.reshape(-1, 1).astype(np.float32))

X_train_iso = np.hstack(iso_feats_train)
X_test_iso = np.hstack(iso_feats_test)

# 4. Combine
X_train_final = np.hstack([X_train_sc, X_train_nca, X_train_iso])
X_test_final = np.hstack([X_test_sc, X_test_nca, X_test_iso])

# Cleanup components to free RAM
del X_train_sc, X_train_nca, X_train_iso
del X_test_sc, X_test_nca, X_test_iso
gc.collect()

print(f"Final Vector Space Dimensions: {X_train_final.shape[1]}")

--- Engineering Vector Space (NCA + Isolation) ---
Fitting NCA (Metric Learning)...
Generating Isolation Embeddings...
Final Vector Space Dimensions: 61


## 5. Training Deep MLP (Optimized)

In [6]:
print("--- Optimization: Hyperparameter Tuning with Cross-Validation ---")

# Grid Search: Checks ALL combinations instead of random ones
param_dist = {
    'hidden_layer_sizes': [(128, 64, 32), (256, 128, 64), (64, 32)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01],
    'batch_size': [64, 128]
}

mlp = MLPClassifier(activation='relu', solver='adam', learning_rate='adaptive', random_state=42, max_iter=300, early_stopping=True)
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Running EXHAUSTIVE Grid Search to find the absolute best model...")
print("NOTE: This may take 5-10 minutes, but it guarantees the best parameters.")

grid_search = GridSearchCV(
    estimator=mlp,
    param_grid=param_dist,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=1, # Sequential to prevent crash
    verbose=3,
)

grid_search.fit(X_train_final, y_train_bal)

print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.2%}")

clf_mlp = grid_search.best_estimator_

--- Optimization: Hyperparameter Tuning with Cross-Validation ---
Running EXHAUSTIVE Grid Search to find the absolute best model...
NOTE: This may take 5-10 minutes, but it guarantees the best parameters.
Fitting 3 folds for each of 36 candidates, totalling 108 fits
[CV 1/3] END alpha=0.0001, batch_size=64, hidden_layer_sizes=(128, 64, 32), learning_rate_init=0.001;, score=0.995 total time=   9.0s
[CV 2/3] END alpha=0.0001, batch_size=64, hidden_layer_sizes=(128, 64, 32), learning_rate_init=0.001;, score=0.995 total time=   8.2s
[CV 3/3] END alpha=0.0001, batch_size=64, hidden_layer_sizes=(128, 64, 32), learning_rate_init=0.001;, score=0.994 total time=   5.0s
[CV 1/3] END alpha=0.0001, batch_size=64, hidden_layer_sizes=(128, 64, 32), learning_rate_init=0.01;, score=0.986 total time=   3.9s
[CV 2/3] END alpha=0.0001, batch_size=64, hidden_layer_sizes=(128, 64, 32), learning_rate_init=0.01;, score=0.988 total time=   4.5s
[CV 3/3] END alpha=0.0001, batch_size=64, hidden_layer_sizes=(128

In [7]:
def predict_risk_sensitive(model, X, encoder, risk_factors):
    probs = model.predict_proba(X)
    classes = encoder.classes_
    
    for cls, factor in risk_factors.items():
        if cls in classes:
            idx = np.where(classes == cls)[0][0]
            probs[:, idx] *= factor
            
    pred_indices = np.argmax(probs, axis=1)
    return encoder.inverse_transform(pred_indices)

RISK_FACTORS = {
    'r2l': 3.0,  
    'u2r': 3.0,  
    'probe': 1.2
}

le_final = LabelEncoder()
le_final.fit(y_vec) # Fit on transformed y_vec

# --- NSL-KDD Validation ---
print("--- Loading NSL-KDD for Validation ---")
NSL_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"
NSL_COLS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

try:
    # Load
    df_nsl = pd.read_csv(NSL_URL, header=None, names=NSL_COLS)
    df_nsl.drop('difficulty_level', axis=1, inplace=True)
    df_nsl['label'] = df_nsl['label'].astype(str).str.replace('.', '', regex=False)
    df_nsl['category'] = df_nsl['label'].map(attack_map).fillna('other')
    
    X_nsl = df_nsl.drop(['label', 'category'], axis=1)
    y_nsl = df_nsl['category']
    
    # 1. Vectorize (Same Encoders)
    X_nsl_vec, _ = prepare_vector_input(X_nsl, fit=False, encoders=encoders)
    
    # 2. Transform Pipeline
    # A. Scale
    X_nsl_sc = scaler.transform(X_nsl_vec).astype(np.float32)
    # B. NCA Project
    X_nsl_nca = nca.transform(X_nsl_sc).astype(np.float32)
    # C. Isolation Embeddings
    # RE-GENERATE ISO SCORES using saved models
    iso_feats_nsl_final = []
    for cls in unique_classes:
         if cls in iso_models:
             iso = iso_models[cls]
             iso_feats_nsl_final.append(iso.decision_function(X_nsl_sc).reshape(-1, 1).astype(np.float32))
            
    X_nsl_iso = np.hstack(iso_feats_nsl_final)

    # D. Stack
    X_nsl_final = np.hstack([X_nsl_sc, X_nsl_nca, X_nsl_iso])
    
    # 3. Predict
    y_nsl_pred = predict_risk_sensitive(clf_mlp, X_nsl_final, le_final, RISK_FACTORS)
    
    # Evaluate
    acc = accuracy_score(y_nsl, y_nsl_pred)
    print(f"\n>>> ENGINEERED VECTOR SPACE ACCURACY (NSL-KDD): {acc:.2%}")
    print("\nClassification Report:")
    print(classification_report(y_nsl, y_nsl_pred))
    
    labels = sorted(y_nsl.unique())
    cm = confusion_matrix(y_nsl, y_nsl_pred, labels=labels)
    plt.figure(figsize=(10, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='magma', xticklabels=labels, yticklabels=labels)
    plt.title("Performance with NCA + Isolation Embeddings")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

except Exception as e:
    print(f"Error: {e}")

--- Loading NSL-KDD for Validation ---

>>> ENGINEERED VECTOR SPACE ACCURACY (NSL-KDD): 0.00%

Classification Report:
Error: Mix of label input types (string and number)
